# psutil 学习笔记（入门速通）

> 目标：掌握 psutil 的系统资源监控、进程信息获取、进程遍历与异常处理。

psutil（process and system utilities）是跨平台的系统监控库，支持 Linux / Windows / macOS / BSD / Solaris / AIX[citation:7]。它的核心价值在于：**用统一的 Python API 替代一堆系统命令**（`ps`、`top`、`free`、`netstat`、`lsof`、`iotop` 等），适合写监控脚本、资源看板、进程管理工具。

**环境准备**
```bash
pip install psutil
```

In [1]:
import psutil
import os
print("psutil version:", psutil.__version__)
print("当前 PID:", os.getpid())

psutil version: 7.2.2
当前 PID: 26776


## 1. CPU 信息

CPU 相关函数分三类：**使用率**、**静态信息**、**时间统计**。

| 函数 | 作用 | 关键说明 |
|---|---|---|
| `cpu_percent(interval=None, percpu=False)` | CPU 使用率（0~100） | `interval>0` 时阻塞采样，结果更准；`interval=None` 返回自上次调用以来的瞬时值，**第一次调用返回无意义的 0.0**，需忽略[citation:3] |
| `cpu_count(logical=True)` | CPU 核心数 | `logical=False` 返回物理核数（排除超线程）[citation:2] |
| `cpu_freq()` | CPU 频率 | 返回 `current`、`min`、`max`（单位 MHz） |
| `cpu_times()` | CPU 时间分解 | 返回 user / system / idle / iowait 等字段（累计秒数） |
| `cpu_stats()` | 上下文切换、中断统计 | `ctx_switches`、`interrupts` 等 |
| `getloadavg()` | 系统负载（1/5/15 分钟） | Windows 上为模拟值 |

**`percpu=True`**：返回每个逻辑核的独立使用率列表，适合看多核负载分布。

**易错点**
- `cpu_percent()` 第一次调用（`interval=None`）返回 0.0，这是设计行为，不是 bug。要得到有意义的数值，要么指定 `interval=1`，要么先调一次丢弃结果、再调一次。
- `interval` 会**阻塞线程**，在交互式环境或 GUI 中使用时注意体验。

In [2]:
print("逻辑核数:", psutil.cpu_count())
print("物理核数:", psutil.cpu_count(logical=False))
print("CPU 频率:", psutil.cpu_freq())

# 阻塞采样 1 秒，结果可靠
print("总使用率:", psutil.cpu_percent(interval=1))
print("每核使用率:", psutil.cpu_percent(interval=1, percpu=True))

# 时间分解
print("CPU times:", psutil.cpu_times())
print("负载:", psutil.getloadavg())

逻辑核数: 16
物理核数: 8
CPU 频率: scpufreq(current=3201.0, min=0.0, max=3201.0)
总使用率: 0.2
每核使用率: [0.0, 0.0, 1.6, 0.0, 3.1, 1.6, 3.1, 0.0, 0.0, 0.0, 1.6, 0.0, 1.6, 0.0, 1.6, 18.8]
CPU times: scputimes(user=2114.84375, system=1085.4531249999418, idle=371153.21875, interrupt=27.59375, dpc=9.21875)
负载: (0.0, 0.0, 0.0)


## 2. 内存信息

内存分**物理内存**和**交换分区**两块，所有数值单位均为**字节**。

| 函数 | 返回对象 | 关键字段 |
|---|---|---|
| `virtual_memory()` | `svmem` | `total` 总量、`available` 可用（真正可分配给新进程的量）、`percent` 使用率、`used` 已用 |
| `swap_memory()` | `sswap` | `total`、`used`、`free`、`percent`、`sin`/`sout`（换入换出字节数） |

**`available` vs `free`**：`free` 是完全空闲的内存，`available` 是**实际可用的内存**（Linux 上约等于 free + buffers + cached）。监控内存压力时应该看 `available` 和 `percent`，而不是 `free`[citation:13]。

**易错点**：`used + available` 不一定等于 `total`，不同平台的统计口径有差异。

In [3]:
mem = psutil.virtual_memory()
print("总内存:", mem.total / 1024**3, "GB")
print("可用:", mem.available / 1024**3, "GB")
print("使用率:", mem.percent, "%")
print("完整信息:", mem)

swap = psutil.swap_memory()
print("\n交换分区使用率:", swap.percent, "%")
print("换入/换出:", swap.sin, swap.sout)

总内存: 15.840328216552734 GB
可用: 2.735107421875 GB
使用率: 82.7 %
完整信息: svmem(total=17008422912, available=2936799232, percent=82.7, used=14071623680, free=2936799232)

交换分区使用率: 4.9 %
换入/换出: 0 0


## 3. 磁盘信息

磁盘分**分区/使用量**和**IO 统计**两类。

| 函数 | 作用 | 说明 |
|---|---|---|
| `disk_partitions(all=False)` | 列出挂载的分区 | 每个分区含 `device`、`mountpoint`、`fstype`、`opts` |
| `disk_usage(path)` | 指定路径的磁盘使用量 | 返回 `total`、`used`、`free`、`percent` |
| `disk_io_counters(perdisk=False)` | 磁盘 IO 累计统计 | `read_count`、`write_count`、`read_bytes`、`write_bytes` 等 |

**`all=True`**：包含伪文件系统（如 `tmpfs`、`proc`、`sysfs`），默认 `False` 只返回物理磁盘分区。

**IO 统计的用法**：`disk_io_counters` 返回的是**累计值**。要计算“每秒读写速率”，需要在两个时间点各取一次，做差再除以时间间隔。

In [4]:
print("物理分区:")
for part in psutil.disk_partitions():
    print(f"  {part.mountpoint:12} {part.fstype:8} {part.device}")

usage = psutil.disk_usage("/")
print(f"\n根分区: {usage.percent}% 已用 "
      f"({usage.used / 1024**3:.1f}G / {usage.total / 1024**3:.1f}G)")

io = psutil.disk_io_counters()
print(f"\n累计读取: {io.read_bytes / 1024**2:.0f} MB")
print(f"累计写入: {io.write_bytes / 1024**2:.0f} MB")

物理分区:
  C:\          NTFS     C:\
  D:\          NTFS     D:\

根分区: 87.5% 已用 (680.1G / 777.0G)

累计读取: 21266 MB
累计写入: 10877 MB


## 4. 网络信息

| 函数 | 作用 | 说明 |
|---|---|---|
| `net_io_counters(pernic=False)` | 网络 IO 累计统计 | `bytes_sent`、`bytes_recv`、`packets_sent`、`packets_recv`；`pernic=True` 按网卡拆分 |
| `net_connections(kind='inet')` | 系统级套接字连接 | 返回 `sconn` 列表，含 `laddr`、`raddr`、`status`、`pid` |
| `net_if_addrs()` | 网卡地址信息 | 返回各网卡的 IP、掩码、广播地址 |
| `net_if_stats()` | 网卡状态 | `isup`（是否启用）、`speed`（速率）、`mtu` |

**`kind` 参数**：`'inet'`（所有 IPv4/IPv6）、`'tcp'`、`'udp'`、`'unix'`、`'all'` 等[citation:10]。

**权限注意**：列出**其他用户**拥有的连接需要 root 权限，否则那些条目的 `pid` 为 `None`[citation:5]。

In [5]:
net = psutil.net_io_counters(pernic=True)
for nic, io in net.items():
    if io.bytes_recv > 0:
        print(f"{nic}: 收 {io.bytes_recv/1024**2:.1f} MB / 发 {io.bytes_sent/1024**2:.1f} MB")

# 列出 TCP 监听端口（可能需要管理员权限才能看到 pid）
print("\nTCP 监听:")
for conn in psutil.net_connections(kind="tcp"):
    if conn.status == psutil.CONN_LISTEN:
        print(f"  {conn.laddr.ip}:{conn.laddr.port}  pid={conn.pid}")

print("\n网卡地址:")
addrs = psutil.net_if_addrs()
for nic in list(addrs)[:2]:
    print(f"  {nic}: {[a.address for a in addrs[nic]]}")

VMware Network Adapter VMnet1: 收 0.0 MB / 发 0.0 MB
VMware Network Adapter VMnet8: 收 0.0 MB / 发 0.0 MB
WLAN: 收 793.3 MB / 发 149.2 MB

TCP 监听:
  :::49664  pid=1292
  127.0.0.1:9210  pid=10384
  0.0.0.0:49666  pid=3396
  :::1947  pid=6068
  192.168.120.1:8995  pid=14276
  0.0.0.0:445  pid=4
  127.0.0.1:22879  pid=7616
  192.168.86.1:139  pid=4
  127.0.0.1:9003  pid=26776
  192.168.100.100:139  pid=4
  127.0.0.1:13016  pid=5352
  0.0.0.0:49668  pid=5624
  0.0.0.0:8998  pid=14276
  0.0.0.0:3580  pid=6284
  127.0.0.1:9006  pid=27592
  0.0.0.0:902  pid=13396
  127.0.0.1:5283  pid=10384
  127.0.0.1:5354  pid=6032
  :::49665  pid=1108
  127.0.0.1:4000  pid=6504
  127.0.0.1:55869  pid=27592
  0.0.0.0:8999  pid=4
  127.0.0.1:49671  pid=6604
  127.0.0.1:65283  pid=26432
  127.0.0.1:27015  pid=6016
  127.0.0.1:9005  pid=27592
  127.0.0.1:9002  pid=26776
  127.0.0.1:59341  pid=22212
  0.0.0.0:49803  pid=1184
  0.0.0.0:49665  pid=1108
  :::8999  pid=4
  ::1:42050  pid=21804
  127.0.0.1:54533  pid=124

## 5. 进程信息（Process 对象）

通过 `psutil.Process(pid)` 获取单个进程的句柄，然后调用方法查询。`pid=None` 时代表**当前进程**。

**身份与状态**

| 方法 | 返回 | 说明 |
|---|---|---|
| `name()` | 进程名 | Linux 上可能被截断到 15 字符 |
| `exe()` | 可执行文件绝对路径 | 可能为空字符串 |
| `cmdline()` | 命令行参数列表 | 启动命令的完整参数 |
| `status()` | 状态 | `running`、`sleeping`、`zombie` 等 |
| `username()` | 所有者用户名 | 需要权限读取其他用户进程 |
| `create_time()` | 创建时间 | Unix 时间戳 |
| `ppid()` / `parent()` / `children()` | 父/子进程 | `children(recursive=True)` 递归获取所有后代 |

**资源消耗**

| 方法 | 返回 | 说明 |
|---|---|---|
| `cpu_percent(interval=None)` | CPU 使用率 | 可超过 100%（多线程跑在多核上）；`interval=None` 第一次返回 0.0[citation:3] |
| `memory_info()` | `rss`、`vms` | `rss` 是实际物理内存占用（常驻集）[citation:3] |
| `memory_percent()` | 占总内存百分比 | 基于 `rss / total` |
| `num_threads()` | 线程数 | |
| `open_files()` | 打开的文件列表 | 含路径和 fd |
| `connections()` | 该进程的网络连接 | 参数同 `net_connections` |

**易错点**：进程可能在两次调用之间**消失**。例如先 `p.name()` 再 `p.memory_info()`，如果进程在这之间退出，第二次调用会抛 `NoSuchProcess`。

In [6]:
p = psutil.Process()  # 当前进程
print("PID:", p.pid)
print("名称:", p.name())
print("状态:", p.status())
print("可执行文件:", p.exe())
print("命令行:", p.cmdline())
print("父 PID:", p.ppid())

print("\n内存 RSS:", p.memory_info().rss / 1024**2, "MB")
print("内存占比:", p.memory_percent(), "%")
print("线程数:", p.num_threads())

# 注意：第一次 cpu_percent(interval=None) 返回 0.0
p.cpu_percent(interval=None)
print("CPU (二次调用):", p.cpu_percent(interval=0.5), "%")

PID: 26776
名称: python.exe
状态: running
可执行文件: D:\software_tech\anaconda3\envs\python_learn\python.exe
命令行: ['d:\\software_tech\\anaconda3\\envs\\python_learn\\python.exe', '-m', 'ipykernel_launcher', '--f=c:\\Users\\Mrlu\\AppData\\Roaming\\jupyter\\runtime\\kernel-v361a2e44af14e03b93bb97b36d83310236f068bf5.json']
父 PID: 28240

内存 RSS: 78.1875 MB
内存占比: 0.48202903011164255 %
线程数: 15
CPU (二次调用): 0.0 %


## 6. 进程遍历与批量操作

`process_iter()` 是遍历系统所有进程的推荐方式。相比 `psutil.pids()` + 逐个 `Process(pid)`，它做了优化：预取属性、自动跳过已消失的进程。

**`attrs` 参数**：传入属性名列表（如 `['pid', 'name', 'cpu_percent']`），`process_iter` 会**一次性批量获取**，结果存在 `proc.info` 字典里，比逐个调用方法快得多[citation:5]。

**常用场景**
- 找出占用 CPU 最高的前 N 个进程
- 按名称查找进程并获取 PID
- 列出监听端口的进程

**易错点**：`process_iter` 遍历时进程可能退出，直接访问 `proc.info` 通常安全（预取已缓存），但后续再调 `proc.memory_info()` 仍可能抛异常。

In [7]:
# 按内存占用排序，取前 5
procs = []
for proc in psutil.process_iter(['pid', 'name', 'memory_percent']):
    procs.append(proc.info)

procs.sort(key=lambda x: x['memory_percent'] or 0, reverse=True)
print("内存占用 Top 5:")
for info in procs[:5]:
    print(f"  {info['pid']:>6}  {info['name'][:20]:<20} {info['memory_percent']:.2f}%")

# 按名称查找进程
target = "python"
found = [p.info for p in psutil.process_iter(['pid', 'name']) 
         if target in p.info['name'].lower()]
print(f"\n名称包含 '{target}' 的进程: {len(found)} 个")
for info in found[:3]:
    print(f"  pid={info['pid']}  {info['name']}")

内存占用 Top 5:
   26432  Code.exe             6.50%
   28240  Code.exe             4.55%
    2124  Code.exe             3.87%
   12372  Tabbit Browser.exe   3.42%
   12152  explorer.exe         2.51%

名称包含 'python' 的进程: 7 个
  pid=3692  pythonw.exe
  pid=9492  pythonw.exe
  pid=12720  pythonw.exe


## 7. 异常处理

psutil 定义了三个常用异常，处理进程相关操作时**几乎必须捕获**：

| 异常 | 含义 | 触发场景 |
|---|---|---|
| `NoSuchProcess` | 进程不存在 | PID 无效，或进程在操作过程中退出 |
| `AccessDenied` | 权限不足 | 尝试读取其他用户（或系统）进程的敏感信息 |
| `ZombieProcess` | 僵尸进程 | 进程已退出但父进程尚未回收 |

**标准处理模式**
```python
try:
    proc = psutil.Process(pid)
    print(proc.name())
except psutil.NoSuchProcess:
    print("进程不存在")
except psutil.AccessDenied:
    print("权限不足")
except psutil.ZombieProcess:
    print("僵尸进程")
```

**`process_iter` 的自动保护**：它内部会捕获 `NoSuchProcess` 并跳过，但 `AccessDenied` 和 `ZombieProcess` 仍可能从你调用的方法中抛出。

In [8]:
# 演示异常处理
fake_pid = 999999
try:
    p = psutil.Process(fake_pid)
    print(p.name())
except psutil.NoSuchProcess:
    print(f"PID {fake_pid} 不存在，已捕获 NoSuchProcess")

# 安全遍历：跳过无权限的进程
safe_count = 0
for proc in psutil.process_iter(['name']):
    try:
        _ = proc.info['name']
        safe_count += 1
    except (psutil.AccessDenied, psutil.NoSuchProcess):
        pass
print(f"安全遍历到 {safe_count} 个进程")

PID 999999 不存在，已捕获 NoSuchProcess
安全遍历到 323 个进程


## 8. 其他实用功能

| 函数 | 作用 |
|---|---|
| `boot_time()` | 系统启动时间（Unix 时间戳） |
| `users()` | 当前登录用户列表 |
| `sensors_temperatures()` | 温度传感器（Linux/部分硬件） |
| `sensors_battery()` | 电池状态（笔记本） |
| `pid_exists(pid)` | 判断 PID 是否存在 |
| `wait_procs(procs, timeout, callback)` | 等待多个进程结束 |

**`sensors_*` 的可用性**：温度和电池依赖平台和硬件驱动，在虚拟机、容器或部分 Windows 机器上可能返回空或不可用，使用时建议加 `hasattr` 或异常保护。

In [9]:
import datetime
print("启动时间:", datetime.datetime.fromtimestamp(psutil.boot_time()))
print("登录用户:", psutil.users())

try:
    temps = psutil.sensors_temperatures()
    if temps:
        for name, entries in list(temps.items())[:1]:
            print(f"温度传感器 {name}: {entries[0].current}°C")
    else:
        print("温度传感器不可用")
except AttributeError:
    print("当前平台不支持 sensors_temperatures")

启动时间: 2026-10-21 14:06:41.108340
登录用户: [suser(name='Mrlu', terminal=None, host=None, started=1792562815.092334, pid=None)]
当前平台不支持 sensors_temperatures


## 9. 综合练习

1. 写一个函数 `top_cpu(n=3)`，返回当前 CPU 占用最高的 n 个进程（名称 + CPU%）。注意 `cpu_percent` 需要预热。
2. 检查根分区磁盘使用率，超过 80% 时打印警告。
3. 列出所有状态为 `LISTEN` 的 TCP 端口及其对应进程名（pid 为 None 时标注“未知”）。

**参考答案**

In [10]:
def top_cpu(n=3):
    # 预热，让 cpu_percent 有基准
    for p in psutil.process_iter(['cpu_percent']):
        pass
    procs = []
    for p in psutil.process_iter(['pid', 'name', 'cpu_percent']):
        procs.append(p.info)
    procs.sort(key=lambda x: x['cpu_percent'] or 0, reverse=True)
    return procs[:n]

print("CPU Top 3:")
for info in top_cpu(3):
    print(f"  {info['name']:<20} {info['cpu_percent']:.1f}%")

usage = psutil.disk_usage("/")
if usage.percent > 80:
    print(f"⚠ 磁盘使用率 {usage.percent}% 超过 80%")
else:
    print(f"磁盘使用率 {usage.percent}% 正常")

print("\nTCP 监听:")
for conn in psutil.net_connections(kind="tcp"):
    if conn.status == psutil.CONN_LISTEN:
        name = "未知"
        if conn.pid:
            try:
                name = psutil.Process(conn.pid).name()
            except (psutil.NoSuchProcess, psutil.AccessDenied):
                pass
        print(f"  {conn.laddr.ip}:{conn.laddr.port:<6} pid={conn.pid} ({name})")

CPU Top 3:
  System Idle Process  1476.6%
  sppsvc.exe           31.9%
  python.exe           25.0%
⚠ 磁盘使用率 87.5% 超过 80%

TCP 监听:
  :::49664  pid=1292 (lsass.exe)
  127.0.0.1:9210   pid=10384 (QQ.exe)
  0.0.0.0:49666  pid=3396 (svchost.exe)
  :::1947   pid=6068 (hasplms.exe)
  192.168.120.1:8995   pid=14276 (SWVisualize.BoostService.exe)
  0.0.0.0:445    pid=4 (System)
  127.0.0.1:22879  pid=7616 (GameViewerServer.exe)
  192.168.86.1:139    pid=4 (System)
  127.0.0.1:9003   pid=26776 (python.exe)
  192.168.100.100:139    pid=4 (System)
  127.0.0.1:13016  pid=5352 (WeChat.exe)
  0.0.0.0:49668  pid=5624 (spoolsv.exe)
  0.0.0.0:8998   pid=14276 (SWVisualize.BoostService.exe)
  0.0.0.0:3580   pid=6284 (SystemWebServer.exe)
  127.0.0.1:9006   pid=27592 (python.exe)
  0.0.0.0:902    pid=13396 (vmware-authd.exe)
  127.0.0.1:5283   pid=10384 (QQ.exe)
  127.0.0.1:5354   pid=6032 (mDNSResponder.exe)
  :::49665  pid=1108 (wininit.exe)
  127.0.0.1:4000   pid=6504 (FoxitProtect.exe)
  127.0.0.1:558

## 常见坑点总结

- **`cpu_percent(interval=None)` 第一次调用返回 0.0**，这是设计行为；要么指定 `interval`，要么先调一次丢弃、再调一次取结果。
- **进程可能在两次调用之间消失**：对单个 `Process` 对象做连续操作时，用 `try/except psutil.NoSuchProcess` 包裹。
- **`net_connections()` 的 `pid` 可能为 None**：读取其他用户的连接需要 root 权限。
- **`disk_io_counters` / `net_io_counters` 返回累计值**，算速率需要两次采样做差。
- **`virtual_memory()` 看 `available` 而不是 `free`**：`free` 是完全空闲，`available` 才是实际可用的内存。
- **`process_iter(attrs=[...])` 比逐个调方法快**：批量预取属性，适合遍历大量进程的场景。
- **`sensors_temperatures()` / `sensors_battery()` 不是所有平台都可用**，使用前加保护。
- **`AccessDenied` 要单独处理**：读取系统进程（如 PID 1）的信息时经常触发。